In [ ]:
# ── imports ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import gaussian_filter
from skimage.feature import peak_local_max
import random, copy, math, os

matplotlib.rcParams['animation.embed_limit'] = 200.0

# ══════════════════════════════════════════════════════════════════
# 1.  PARAMETERS
# ══════════════════════════════════════════════════════════════════
IMAGE_SIZE    = 256
MIN_RADIUS    = 6
MAX_RADIUS    = 14
NUM_PARTICLES = 60

NUM_IMAGES   = 2000
BATCH_SIZE   = 8
MAX_EPOCHS   = 60
PATIENCE     = 8

HMAP_W   = 1.0
MASK_W   = 1.0
OFFSET_W = 0.5          # weight for offset head loss

DETECT_THRESHOLD = 0.15
NMS_MIN_DIST     = 3    # Lowered from MIN_RADIUS to safely allow touching particles

HARD_EXAMPLE_FRAC = 0.30

# Multi-type rendering fractions (must sum to 1.0)
RENDER_TYPE_PROBS = {
    'phase_contrast': 0.50,
    'darkfield':      0.20,
    'fluorescence':   0.15,
    'dic':            0.10,
    'brightfield':    0.05,
}

OUT_DIR = os.path.join(os.getcwd(), "colloid_output")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Device / multi-GPU setup ──────────────────────────────────────
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_GPUS = torch.cuda.device_count() if device.type == 'cuda' else 0
USE_AMP  = device.type == 'cuda'

if NUM_GPUS > 1:
    BATCH_SIZE = BATCH_SIZE * NUM_GPUS

print(f"Device : {device}  |  GPUs detected: {NUM_GPUS}  |  AMP: {USE_AMP}")
print(f"Effective batch size: {BATCH_SIZE}")
print(f"Output directory: {OUT_DIR}")


# ══════════════════════════════════════════════════════════════════
# 2.  MULTI-TYPE MICROSCOPY RENDERER  (layer-based, correct overlap)
# ══════════════════════════════════════════════════════════════════
def _make_coords(cx, cy, r, img_shape, pad_factor=4.0):
    """Return (x0,x1,y0,y1, rr, rn) for a particle patch."""
    pad  = int(r * pad_factor)
    H, W = img_shape
    x0, x1 = max(0, cx - pad), min(W, cx + pad + 1)
    y0, y1 = max(0, cy - pad), min(H, cy + pad + 1)
    if x1 <= x0 or y1 <= y0:
        return None
    xs = np.arange(x0, x1, dtype=np.float32) - cx
    ys = np.arange(y0, y1, dtype=np.float32) - cy
    xx, yy = np.meshgrid(xs, ys)
    rr = np.sqrt(xx**2 + yy**2)
    rn = rr / r
    return x0, x1, y0, y1, xx, yy, rr, rn


def _apply_asymmetry(xx, yy, asymmetry, asymmetry_angle):
    if asymmetry <= 0.0:
        return np.sqrt(xx**2 + yy**2)
    cos_a = math.cos(asymmetry_angle)
    sin_a = math.sin(asymmetry_angle)
    xr =  xx * cos_a + yy * sin_a
    yr = -xx * sin_a + yy * cos_a
    xr *= (1.0 + asymmetry)
    xx_eff = xr * cos_a - yr * sin_a
    yy_eff = xr * sin_a + yr * cos_a
    return np.sqrt(xx_eff**2 + yy_eff**2)


def render_particle_layer(image_shape, cx, cy, r, intensity, defocus,
                           asymmetry=0.0, asymmetry_angle=0.0,
                           secondary_fringe_amp=0.0,
                           render_type='phase_contrast'):
    """
    Render a single particle into its OWN blank layer (ones = transparent).
    Returns (layer, x0, x1, y0, y1) or None if out of bounds.
    The caller composites multiple layers correctly.
    """
    coords = _make_coords(cx, cy, r, image_shape)
    if coords is None:
        return None

    x0, x1, y0, y1, xx, yy, rr, rn = coords
    layer = np.ones(image_shape, dtype=np.float32)

    if render_type == 'phase_contrast':
        rr_eff = _apply_asymmetry(xx, yy, asymmetry, asymmetry_angle)
        rn_eff = rr_eff / r

        dark_thickness = 0.12 + 0.15 * defocus
        dark_ring      = -0.8 * intensity * np.exp(-((rn_eff - 0.95)**2) /
                                                    (2 * dark_thickness**2))
        bright_center  = 0.6 * intensity * (1.0 - defocus * 0.3) * \
                         np.exp(-(rn_eff**2) / (2 * 0.25**2))
        fringe_freq    = 3.0 + 1.5 * defocus
        fringe_decay   = 0.3 + 0.2 * defocus
        fringes        = (0.2 * intensity
                          * np.cos((rn_eff - 1.0) * fringe_freq * math.pi)
                          * np.exp(-(rn_eff - 1.0) / fringe_decay))
        fringes        = np.where(rn_eff > 1.0, fringes, 0.0)
        transmission   = (1.0 + dark_ring + bright_center + fringes
                          + secondary_fringe_amp * 0.1 * fringes)
        transmission   = np.clip(transmission, 0.05, 2.5)
        edge_blend     = np.clip((4.0 - rn_eff) / 0.8, 0.0, 1.0)
        transmission   = 1.0 + (transmission - 1.0) * edge_blend
        roi_mask       = rn_eff < 4.0

    elif render_type == 'darkfield':
        bright_ring  = intensity * np.exp(-((rn - 0.95)**2) / (2 * 0.10**2))
        halo         = 0.3 * intensity * np.exp(-(rn**2) / (2 * 1.5**2))
        transmission = 1.0 + bright_ring + halo * np.where(rn > 1.0, 1.0, 0.0)
        transmission = np.clip(transmission, 1.0, 4.0)
        roi_mask     = rn < 3.0

    elif render_type == 'fluorescence':
        sigma_psf    = r / 2.5 + defocus * r * 0.4
        blob         = intensity * np.exp(-(rr**2) / (2 * sigma_psf**2))
        transmission = 1.0 + blob
        roi_mask     = rr < sigma_psf * 3.0

    elif render_type == 'dic':
        # Differential interference contrast: shadow-relief effect
        sigma_dic  = r * 0.6
        gauss      = np.exp(-(rr**2) / (2 * sigma_dic**2))
        # gradient along shear direction
        shear_cos  = math.cos(asymmetry_angle)
        shear_sin  = math.sin(asymmetry_angle)
        dx         = -xx / (sigma_dic**2) * gauss
        dy         = -yy / (sigma_dic**2) * gauss
        relief     = intensity * (shear_cos * dx + shear_sin * dy)
        relief     = relief / (np.abs(relief).max() + 1e-8) * 0.6 * intensity
        transmission = 1.0 + relief
        roi_mask   = rn < 3.0

    elif render_type == 'brightfield':
        # Low-contrast transmission dip
        dip          = -0.25 * intensity * np.exp(-(rr**2) / (2 * (r * 0.5)**2))
        rim          = 0.10 * intensity * np.exp(-((rn - 1.0)**2) / (2 * 0.15**2))
        transmission = 1.0 + dip + rim
        transmission = np.clip(transmission, 0.6, 1.3)
        roi_mask     = rn < 2.5

    else:
        raise ValueError(f"Unknown render_type: {render_type}")

    patch = layer[y0:y1, x0:x1]
    patch[roi_mask] = np.where(
        roi_mask,
        transmission,
        patch
    )[roi_mask]
    layer[y0:y1, x0:x1] = patch
    return layer, x0, x1, y0, y1


def composite_layers(layers, background, render_type):
    """
    Composite particle layers onto background correctly for each modality.
    For multiplicative types: take element-wise MINIMUM to preserve saddle
    points between overlapping particles (each particle darkens independently).
    For additive types (fluorescence, darkfield): sum contributions.
    """
    img = background.copy()

    additive_types = {'fluorescence', 'darkfield'}

    if render_type in additive_types:
        for layer, *_ in layers:
            # layer encodes additive brightness above 1.0
            img += (layer - 1.0)
    else:
        # Build a per-pixel minimum-transmission map across all particles.
        # This correctly creates saddle points where two dark rings overlap.
        combined_delta = np.zeros_like(img)  # sum of (layer-1) per particle
        for layer, *_ in layers:
            combined_delta += (layer - 1.0)
        img += combined_delta

    return img


# ══════════════════════════════════════════════════════════════════
# 3.  GROUND-TRUTH HELPERS
# ══════════════════════════════════════════════════════════════════
def render_gaussian_additive(hmap, cy, cx, sigma):
    """Additive Gaussian heatmap — preserves two peaks for overlapping particles."""
    H, W   = hmap.shape
    radius = int(math.ceil(3 * sigma))
    x0, x1 = max(0, cx - radius), min(W, cx + radius + 1)
    y0, y1 = max(0, cy - radius), min(H, cy + radius + 1)
    xs = np.arange(x0, x1) - cx
    ys = np.arange(y0, y1) - cy
    xx, yy = np.meshgrid(xs, ys)
    g = np.exp(-(xx**2 + yy**2) / (2 * sigma**2)).astype(np.float32)
    hmap[y0:y1, x0:x1] += g          # ADDITIVE — not np.maximum


def render_offset_map(offset_map, weight_map, cy, cx, r):
    """
    For each pixel within radius r of (cx,cy), store the offset
    vector pointing toward the center in exact pixels.
    weight_map tracks which center is closest.
    offset_map: (2, H, W)  — channel 0 = dy, channel 1 = dx
    """
    H, W   = offset_map.shape[1], offset_map.shape[2]
    pad    = int(r * 1.5)
    x0, x1 = max(0, cx - pad), min(W, cx + pad + 1)
    y0, y1 = max(0, cy - pad), min(H, cy + pad + 1)
    xs = np.arange(x0, x1) - cx
    ys = np.arange(y0, y1) - cy
    xx, yy = np.meshgrid(xs, ys)
    rr     = np.sqrt(xx**2 + yy**2) + 1e-6
    inside = rr < r * 1.5
    
    # dy = (cy - py), dx = (cx - px) in EXACT PIXELS (no longer dividing by r)
    dy_patch = np.where(inside, -yy, 0.0).astype(np.float32)
    dx_patch = np.where(inside, -xx, 0.0).astype(np.float32)
    dist_patch = np.where(inside, rr, np.inf).astype(np.float32)

    # Only overwrite where this center is closer
    existing_dist = weight_map[y0:y1, x0:x1]
    closer = dist_patch < existing_dist
    offset_map[0, y0:y1, x0:x1] = np.where(closer, dy_patch, offset_map[0, y0:y1, x0:x1])
    offset_map[1, y0:y1, x0:x1] = np.where(closer, dx_patch, offset_map[1, y0:y1, x0:x1])
    weight_map[y0:y1, x0:x1] = np.where(closer, dist_patch, existing_dist)


def adaptive_sigma(r):
    return r / 2.5          # was r/3 — less punishing for slight spatial errors


# ══════════════════════════════════════════════════════════════════
# 4.  BACKGROUND GENERATION  (diverse illumination)
# ══════════════════════════════════════════════════════════════════
def make_background(image_size, render_type):
    gx, gy = np.meshgrid(np.linspace(0, 1, image_size),
                          np.linspace(0, 1, image_size))

    if render_type in ('fluorescence', 'darkfield'):
        bg = np.full((image_size, image_size),
                     random.uniform(0.02, 0.08), dtype=np.float32)
    elif render_type == 'dic':
        bg = np.full((image_size, image_size),
                     random.uniform(0.45, 0.55), dtype=np.float32)
    else:
        bg = (0.60 + 0.15 * gx + 0.10 * gy).astype(np.float32)
        angle = random.uniform(0, math.pi)
        bg = (random.uniform(0.50, 0.70)
              + random.uniform(0.05, 0.20) * (gx * math.cos(angle)
                                               + gy * math.sin(angle))
              ).astype(np.float32)

    noise_scale = random.uniform(0.03, 0.15)
    bg_noise = gaussian_filter(
        np.random.randn(image_size, image_size).astype(np.float32),
        sigma=random.uniform(20, 60)
    )
    bg_noise_range = bg_noise.max() - bg_noise.min()
    if bg_noise_range > 0:
        bg_noise = (bg_noise - bg_noise.min()) / bg_noise_range
    bg += noise_scale * bg_noise

    cy_v, cx_v = image_size / 2.0, image_size / 2.0
    dist_fc = np.sqrt((gx * image_size - cx_v)**2 +
                      (gy * image_size - cy_v)**2)
    max_d = dist_fc.max() + 1e-8
    vignette_str = random.uniform(0.0, 0.25)
    vignette = 1.0 - vignette_str * (dist_fc / max_d)
    bg *= vignette.astype(np.float32)

    return np.clip(bg, 0.0, 1.0)


# ══════════════════════════════════════════════════════════════════
# 5.  SYNTHETIC SAMPLE GENERATION
# ══════════════════════════════════════════════════════════════════
def _sample_render_type():
    types = list(RENDER_TYPE_PROBS.keys())
    probs = list(RENDER_TYPE_PROBS.values())
    return random.choices(types, weights=probs, k=1)[0]


def generate_sample(image_size, num_particles, min_r, max_r, force_touching=False):
    render_type = _sample_render_type()

    hmap       = np.zeros((image_size, image_size), dtype=np.float32)
    msk        = np.zeros((image_size, image_size), dtype=np.float32)
    offset_map = np.zeros((2, image_size, image_size), dtype=np.float32)
    weight_map = np.full((image_size, image_size), np.inf, dtype=np.float32)

    bg  = make_background(image_size, render_type)

    placed  = []   # (cx, cy, r)
    centers = []   # (cx, cy) for output

    particle_layers = []

    for i in range(num_particles):
        r       = random.randint(min_r, max_r)
        defocus = random.uniform(0.15, 0.95)
        intens  = random.uniform(0.60, 1.0)
        asymmetry       = random.uniform(0.0, 0.15)
        asymmetry_angle = random.uniform(0.0, math.pi)
        secondary_amp   = random.uniform(0.10, 0.40) if random.random() < 0.50 else 0.0

        max_tries = 30
        for _ in range(max_tries):
            if force_touching and len(placed) > 0:
                ref_cx, ref_cy, ref_r = random.choice(placed)
                angle = random.uniform(0, 2 * math.pi)
                dist  = ref_r + r + random.randint(0, 3)
                cx    = int(np.clip(ref_cx + dist * math.cos(angle),
                                    max_r + 1, image_size - max_r - 2))
                cy    = int(np.clip(ref_cy + dist * math.sin(angle),
                                    max_r + 1, image_size - max_r - 2))
            else:
                cx = random.randint(max_r + 1, image_size - max_r - 2)
                cy = random.randint(max_r + 1, image_size - max_r - 2)

            # Allow partial overlaps but reject near-complete merges
            min_sep = (r + (placed[-1][2] if placed else r)) * 0.5
            too_close = any(
                math.hypot(cx - px, cy - py) < (r + pr) * 0.45
                for px, py, pr in placed
            )
            if not too_close:
                break
        else:
            continue

        placed.append((cx, cy, r))
        centers.append((cx, cy))

        result = render_particle_layer(
            (image_size, image_size), cx, cy, r, intens, defocus,
            asymmetry=asymmetry,
            asymmetry_angle=asymmetry_angle,
            secondary_fringe_amp=secondary_amp,
            render_type=render_type
        )
        if result is not None:
            particle_layers.append(result)

        sigma = adaptive_sigma(r)
        render_gaussian_additive(hmap, cy, cx, sigma)
        cv2.circle(msk, (cx, cy), r, 1.0, -1)
        render_offset_map(offset_map, weight_map, cy, cx, r)

    img = composite_layers(particle_layers, bg, render_type)
    img = cv2.GaussianBlur(img, (3, 3), 0.8)

    photon_scale = random.uniform(200.0, 600.0)
    photons = np.clip(img * photon_scale, 0, 1e6).astype(np.float64)
    img     = np.random.poisson(photons).astype(np.float32) / photon_scale
    img += np.random.normal(0, 0.015, img.shape).astype(np.float32)

    num_hot = np.random.poisson(5)
    if num_hot > 0:
        hy = np.random.randint(0, image_size, num_hot)
        hx = np.random.randint(0, image_size, num_hot)
        img[hy, hx] = random.uniform(0.8, 1.0)

    if random.random() < 0.3:
        band_amp  = random.uniform(0.01, 0.04)
        band_freq = random.uniform(0.05, 0.2)
        bands     = band_amp * np.sin(
            2 * math.pi * band_freq * np.arange(image_size)
        ).astype(np.float32)
        img += bands[:, None]

    img = np.clip(img, 0, None)
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo)

    hmap = np.clip(hmap, 0.0, 1.0)
    msk  = np.clip(msk,  0.0, 1.0)

    return img[None], hmap[None], msk[None], offset_map, centers


# ══════════════════════════════════════════════════════════════════
# 6.  DATASET
# ══════════════════════════════════════════════════════════════════
print("Generating synthetic dataset ...")
all_imgs, all_hmaps, all_masks, all_offsets, all_centers = [], [], [], [], []

for i in range(NUM_IMAGES):
    force = random.random() < HARD_EXAMPLE_FRAC
    im, hm, mk, off, ctr = generate_sample(
        IMAGE_SIZE, NUM_PARTICLES, MIN_RADIUS, MAX_RADIUS,
        force_touching=force
    )
    all_imgs.append(im)
    all_hmaps.append(hm)
    all_masks.append(mk)
    all_offsets.append(off)
    all_centers.append(ctr)
    if (i + 1) % 400 == 0:
        print(f"  {i+1}/{NUM_IMAGES}")

all_imgs    = np.stack(all_imgs).astype(np.float32)
all_hmaps   = np.stack(all_hmaps).astype(np.float32)
all_masks   = np.stack(all_masks).astype(np.float32)
all_offsets = np.stack(all_offsets).astype(np.float32)

t1, t2 = int(NUM_IMAGES * 0.70), int(NUM_IMAGES * 0.85)

class ColloidDataset(Dataset):
    def __init__(self, imgs, hmaps, masks, offsets, centers, augment=False):
        self.imgs    = imgs
        self.hmaps   = hmaps
        self.masks   = masks
        self.offsets = offsets
        self.centers = centers
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img = self.imgs[idx].copy()
        hm  = self.hmaps[idx].copy()
        mk  = self.masks[idx].copy()
        off = self.offsets[idx].copy()

        if self.augment:
            if random.random() > 0.5:
                img = np.flip(img, axis=2).copy()
                hm  = np.flip(hm,  axis=2).copy()
                mk  = np.flip(mk,  axis=2).copy()
                off = np.flip(off, axis=2).copy()
                off[1] = -off[1]

            if random.random() > 0.5:
                img = np.flip(img, axis=1).copy()
                hm  = np.flip(hm,  axis=1).copy()
                mk  = np.flip(mk,  axis=1).copy()
                off = np.flip(off, axis=1).copy()
                off[0] = -off[0]

            k = random.randint(0, 3)
            if k > 0:
                img = np.rot90(img, k, axes=(1, 2)).copy()
                hm  = np.rot90(hm,  k, axes=(1, 2)).copy()
                mk  = np.rot90(mk,  k, axes=(1, 2)).copy()
                for _ in range(k):
                    dy_old = off[0].copy()
                    dx_old = off[1].copy()
                    off[0] = -dx_old
                    off[1] =  dy_old
                off = np.rot90(off, k, axes=(1, 2)).copy()

            img = np.clip(img + random.uniform(-0.07, 0.07), 0, 1).astype(np.float32)
            mean = img.mean()
            img  = np.clip(
                (img - mean) * random.uniform(0.85, 1.15) + mean, 0, 1
            ).astype(np.float32)

        return (torch.from_numpy(img),
                torch.from_numpy(hm),
                torch.from_numpy(mk),
                torch.from_numpy(off))

def make_loader(start, stop, aug):
    ds = ColloidDataset(
        all_imgs[start:stop],
        all_hmaps[start:stop],
        all_masks[start:stop],
        all_offsets[start:stop],
        all_centers[start:stop],
        augment=aug
    )
    nw = min(4, os.cpu_count() or 1) if NUM_GPUS > 1 else 0
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=aug,
                      num_workers=nw, pin_memory=(device.type == 'cuda'),
                      persistent_workers=(nw > 0))

train_loader = make_loader(0,  t1, True)
val_loader   = make_loader(t1, t2, False)
test_loader  = make_loader(t2, NUM_IMAGES, False)


# ══════════════════════════════════════════════════════════════════
# 7.  ARCHITECTURE
# ══════════════════════════════════════════════════════════════════
class ConvBnRelu(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, p=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, k, padding=p, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class ResBlock(nn.Module):
    def __init__(self, ch, drop=0.10):
        super().__init__()
        self.bn1   = nn.BatchNorm2d(ch)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(ch)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.drop  = drop

    def forward(self, x):
        h = self.conv1(F.relu(self.bn1(x)))
        h = F.dropout(h, self.drop, self.training)
        h = self.conv2(F.relu(self.bn2(h)))
        return x + h


class AttentionGate(nn.Module):
    def __init__(self, f_g, f_x, f_int):
        super().__init__()
        self.W_g = nn.Conv2d(f_g, f_int, 1, bias=False)
        self.W_x = nn.Conv2d(f_x, f_int, 1, bias=False)
        self.psi = nn.Conv2d(f_int, 1, 1, bias=True)
        self.bn  = nn.BatchNorm2d(1)

    def forward(self, g, x):
        g_up = F.interpolate(g, size=x.shape[2:], mode='bilinear', align_corners=False)
        attn = torch.sigmoid(self.bn(self.psi(F.relu(self.W_g(g_up) + self.W_x(x)))))
        return x * attn


class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = ConvBnRelu(in_ch, out_ch)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='bilinear', align_corners=False))


class ASPP(nn.Module):
    def __init__(self, in_ch, out_ch, dilations=(1, 2, 4, 8)):
        super().__init__()
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 3, padding=d, dilation=d, bias=False),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
            )
            for d in dilations
        ])
        self.gap = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_ch * (len(dilations) + 1), out_ch, 1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Dropout(0.10),
        )

    def forward(self, x):
        feats = [b(x) for b in self.branches]
        gap   = F.interpolate(self.gap(x), size=x.shape[2:], mode='bilinear', align_corners=False)
        feats.append(gap)
        return self.project(torch.cat(feats, dim=1))


class OptimalColloidNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(ConvBnRelu(1, 32), ResBlock(32))
        self.enc1 = nn.Sequential(nn.MaxPool2d(2), ConvBnRelu(32,  64),  ResBlock(64))
        self.enc2 = nn.Sequential(nn.MaxPool2d(2), ConvBnRelu(64,  128), ResBlock(128), ResBlock(128))
        self.enc3 = nn.Sequential(nn.MaxPool2d(2), ConvBnRelu(128, 256), ResBlock(256), ResBlock(256))
        self.enc4 = nn.Sequential(nn.MaxPool2d(2), ConvBnRelu(256, 512),
                                  ResBlock(512, drop=0.15), ResBlock(512, drop=0.15))

        self.bottleneck = ASPP(512, 512, dilations=[1, 2, 4, 8])

        self.ag4 = AttentionGate(512, 256, 128)
        self.ag3 = AttentionGate(256, 128,  64)
        self.ag2 = AttentionGate(128,  64,  32)
        self.ag1 = AttentionGate( 64,  32,  16)

        self.up4  = UpBlock(512, 256)
        self.dec4 = nn.Sequential(ConvBnRelu(512, 256), ResBlock(256))
        self.up3  = UpBlock(256, 128)
        self.dec3 = nn.Sequential(ConvBnRelu(256, 128), ResBlock(128))
        self.up2  = UpBlock(128, 64)
        self.dec2 = nn.Sequential(ConvBnRelu(128, 64),  ResBlock(64))
        self.up1  = UpBlock(64, 32)
        self.dec1 = nn.Sequential(ConvBnRelu(64, 32),   ResBlock(32))

        self.refine_hmap   = nn.Sequential(ConvBnRelu(32, 32), ResBlock(32), ConvBnRelu(32, 32))
        self.refine_mask   = nn.Sequential(ConvBnRelu(32, 32), ResBlock(32), ConvBnRelu(32, 32))
        self.refine_offset = nn.Sequential(ConvBnRelu(32, 32), ResBlock(32), ConvBnRelu(32, 32))

        self.heatmap_head = nn.Sequential(ConvBnRelu(32, 32), nn.Conv2d(32, 1, 1))
        nn.init.constant_(self.heatmap_head[-1].bias, -2.19)

        self.mask_head = nn.Sequential(ConvBnRelu(32, 32), nn.Conv2d(32, 1, 1))
        nn.init.constant_(self.mask_head[-1].bias, -2.19)

        # Removed bounding tanh activation. Outputs direct pixel displacements
        self.offset_head = nn.Sequential(ConvBnRelu(32, 32), nn.Conv2d(32, 2, 1))

    def forward(self, x):
        s  = self.stem(x)
        e1 = self.enc1(s)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        b  = self.bottleneck(e4)

        d4 = self.dec4(torch.cat([self.up4(b),  self.ag4(b,  e3)], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), self.ag3(d4, e2)], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), self.ag2(d3, e1)], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), self.ag1(d2, s)],  1))

        hmap_out   = self.heatmap_head(self.refine_hmap(d1))
        mask_out   = self.mask_head(self.refine_mask(d1))
        offset_out = self.offset_head(self.refine_offset(d1)) 

        return hmap_out, mask_out, offset_out


_model = OptimalColloidNet().to(device)
if NUM_GPUS > 1: model = nn.DataParallel(_model)
else: model = _model


# ══════════════════════════════════════════════════════════════════
# 8.  LOSS FUNCTIONS
# ══════════════════════════════════════════════════════════════════
def focal_loss(pred_logits, targets, alpha=2.0, beta=4.0):
    pred_logits = pred_logits.float()
    targets     = targets.float()
    pred        = torch.sigmoid(pred_logits).clamp(1e-5, 1.0 - 1e-5)

    pos     = (targets == 1.0).float() # Strict threshold for ground-truth centers
    neg     = 1.0 - pos
    pos_loss = -pos * torch.pow(1.0 - pred, alpha) * torch.log(pred)
    neg_loss = (-neg * torch.pow(1.0 - targets, beta)
                * torch.pow(pred, alpha) * torch.log(1.0 - pred))

    N = pos.sum().clamp(min=1.0)
    return (pos_loss.sum() + neg_loss.sum()) / N


def dice_bce_loss(pred_logits, targets, smooth=1.0):
    pred_logits = pred_logits.float()
    targets     = targets.float()
    bce         = F.binary_cross_entropy_with_logits(pred_logits, targets)
    pred        = torch.sigmoid(pred_logits)
    inter       = (pred * targets).sum(dim=(1, 2, 3))
    dice        = 1.0 - (2.0 * inter + smooth) / (
        pred.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) + smooth)
    return bce + dice.mean()


def offset_loss(pred_offset, gt_offset):
    pred_offset = pred_offset.float()
    gt_offset   = gt_offset.float()
    mag      = (gt_offset ** 2).sum(dim=1, keepdim=True).sqrt()
    mask     = (mag > 0.01).float()
    N        = mask.sum().clamp(min=1.0)
    loss     = F.smooth_l1_loss(pred_offset * mask, gt_offset * mask, reduction='sum')
    return loss / N


def total_loss(hmap_pred, mask_pred, offset_pred, hmap_gt, mask_gt, offset_gt):
    lh  = focal_loss(hmap_pred, hmap_gt)
    lm  = dice_bce_loss(mask_pred, mask_gt)
    lo  = offset_loss(offset_pred, offset_gt)
    return HMAP_W * lh + MASK_W * lm + OFFSET_W * lo, lh, lm, lo


# ══════════════════════════════════════════════════════════════════
# 9.  DETECTION — using learned offset fields for exact position
# ══════════════════════════════════════════════════════════════════
def detect_centers(hmap, offset_map, threshold=DETECT_THRESHOLD, min_dist=NMS_MIN_DIST):
    """
    Find integer heatmap peaks with NMS, then refine strictly
    using the network's predicted vector displacement field.
    """
    raw_peaks = peak_local_max(hmap, min_distance=min_dist,
                                threshold_abs=threshold, exclude_border=False)

    if len(raw_peaks) == 0:
        return np.empty((0, 2), dtype=np.float32)

    refined = []
    for pk in raw_peaks:
        r, c = pk[0], pk[1]
        # Read exact pixel displacement predicted by the network
        # offset_map is (2, H, W) where 0 = dy, 1 = dx
        dy, dx = offset_map[:, r, c] 
        refined.append([r + dy, c + dx])

    return np.array(refined, dtype=np.float32)


# ══════════════════════════════════════════════════════════════════
# 10.  DETECTION METRICS
# ══════════════════════════════════════════════════════════════════
def detection_metrics(centers_pred, centers_gt, tol=NMS_MIN_DIST):
    if len(centers_pred) == 0 and len(centers_gt) == 0: return 1.0, 1.0, 1.0
    if len(centers_pred) == 0: return 0.0, 0.0, 0.0
    if len(centers_gt) == 0:   return 0.0, 1.0, 0.0

    D = cdist(np.array(centers_pred), np.array(centers_gt))
    cost = np.where(D < tol, D, 1e9)
    row_ind, col_ind = linear_sum_assignment(cost)
    tp        = int((D[row_ind, col_ind] < tol).sum())
    precision = tp / len(centers_pred)
    recall    = tp / len(centers_gt)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1


# ══════════════════════════════════════════════════════════════════
# 11.  TRAINING
# ══════════════════════════════════════════════════════════════════
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=20, T_mult=1, eta_min=3e-7
)
scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

best_val     = float('inf')
patience_cnt = 0
best_wts     = copy.deepcopy(_model.state_dict())
best_opt_wts = copy.deepcopy(optimizer.state_dict())
best_epoch   = 1
train_losses, val_losses, val_f1s = [], [], []

val_gt_centers = []
for idx in range(t1, t2):
    if all_centers[idx]:
        val_gt_centers.append(np.array([(cy, cx) for cx, cy in all_centers[idx]], dtype=np.float32))
    else:
        val_gt_centers.append(np.empty((0, 2), dtype=np.float32))

print("\nTraining OptimalColloidNet ...")

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    t_sum = 0.0

    for batch_idx, (imgs, hmaps, masks, offsets) in enumerate(train_loader):
        imgs, hmaps, masks, offsets = (imgs.to(device), hmaps.to(device),
                                       masks.to(device), offsets.to(device))
        optimizer.zero_grad()
        
        # Proper fractional epoch calculation for scheduler
        fractional_epoch = epoch - 1 + (batch_idx / len(train_loader))

        if USE_AMP:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                hp, mp, op = model(imgs)
                loss, lh, lm, lo = total_loss(hp, mp, op, hmaps, masks, offsets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() >= scale_before:
                scheduler.step(fractional_epoch)
        else:
            hp, mp, op = model(imgs)
            loss, lh, lm, lo = total_loss(hp, mp, op, hmaps, masks, offsets)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step(fractional_epoch)

        t_sum += loss.item()

    model.eval()
    v_sum = 0.0
    all_prec, all_rec, all_f1 = [], [], []
    val_sample_idx = 0

    with torch.no_grad():
        for imgs, hmaps, masks, offsets in val_loader:
            imgs, hmaps, masks, offsets = (imgs.to(device), hmaps.to(device),
                                           masks.to(device), offsets.to(device))
            if USE_AMP:
                with torch.autocast(device_type='cuda', dtype=torch.float16):
                    hp, mp, op = model(imgs)
            else:
                hp, mp, op = model(imgs)

            v_sum += total_loss(hp, mp, op, hmaps, masks, offsets)[0].item()

            hmap_np   = torch.sigmoid(hp[:, 0]).cpu().float().numpy()
            offset_np = op.cpu().float().numpy()

            for b in range(hmap_np.shape[0]):
                pred_centers = detect_centers(hmap_np[b], offset_np[b])
                gt_centers   = val_gt_centers[val_sample_idx]
                val_sample_idx += 1
                p, r, f = detection_metrics(pred_centers, gt_centers)
                all_prec.append(p)
                all_rec.append(r)
                all_f1.append(f)

    avg_t  = t_sum / len(train_loader)
    avg_v  = v_sum / len(val_loader)
    avg_f1 = float(np.mean(all_f1)) if all_f1 else 0.0
    train_losses.append(avg_t)
    val_losses.append(avg_v)
    val_f1s.append(avg_f1)
    lr_now = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch:3d}/{MAX_EPOCHS}  train {avg_t:.4f}  val {avg_v:.4f}  "
          f"F1 {avg_f1:.3f}  P {np.mean(all_prec):.3f}  R {np.mean(all_rec):.3f}  lr {lr_now:.2e}")

    if avg_v < best_val:
        best_val     = avg_v
        best_wts     = copy.deepcopy(_model.state_dict())
        best_opt_wts = copy.deepcopy(optimizer.state_dict())
        best_epoch   = epoch
        patience_cnt = 0
    else:
        patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print(f"Early stop at epoch {epoch}. Best val: {best_val:.4f}")
            break

_model.load_state_dict(best_wts)
print(f"Best weights restored (epoch {best_epoch}, val {best_val:.4f}).")
ckpt_path = os.path.join(OUT_DIR, 'best_checkpoint.pt')
torch.save({'epoch': best_epoch, 'model_state': best_wts,
            'optimizer_state': best_opt_wts, 'val_loss': best_val}, ckpt_path)


# ══════════════════════════════════════════════════════════════════
# 12.  INFERENCE WITH TEST-TIME AUGMENTATION (8-fold)
# ══════════════════════════════════════════════════════════════════
def infer_tta(img_tensor):
    _model.eval()
    augmented, aug_params = [], []

    for rot_k in (0, 1):
        for hflip in (False, True):
            for vflip in (False, True):
                t = img_tensor.clone()
                if hflip: t = torch.flip(t, dims=[3])
                if vflip: t = torch.flip(t, dims=[2])
                if rot_k > 0: t = torch.rot90(t, rot_k, dims=[2, 3])
                augmented.append(t)
                aug_params.append((rot_k, hflip, vflip))

    batch = torch.cat(augmented, dim=0)

    with torch.no_grad():
        if USE_AMP:
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                hp_batch, mp_batch, op_batch = _model(batch)
        else:
            hp_batch, mp_batch, op_batch = _model(batch)

    hm_batch = torch.sigmoid(hp_batch[:, 0])
    mk_batch = torch.sigmoid(mp_batch[:, 0])

    aug_hmaps, aug_masks, aug_offsets = [], [], []

    for i, (rot_k, hflip, vflip) in enumerate(aug_params):
        hm, mk, off = hm_batch[i], mk_batch[i], op_batch[i]

        if rot_k > 0:
            hm, mk = torch.rot90(hm, -rot_k, dims=[0, 1]), torch.rot90(mk, -rot_k, dims=[0, 1])
            for _ in range(4 - rot_k):
                off = torch.stack([-off[1].clone(), off[0].clone()], dim=0)
            off = torch.rot90(off, -rot_k, dims=[1, 2])

        if vflip:
            hm, mk, off = torch.flip(hm, [0]), torch.flip(mk, [0]), torch.flip(off, [1])
            off[0] = -off[0]

        if hflip:
            hm, mk, off = torch.flip(hm, [1]), torch.flip(mk, [1]), torch.flip(off, [2])
            off[1] = -off[1]

        aug_hmaps.append(hm)
        aug_masks.append(mk)
        aug_offsets.append(off)

    return (torch.stack(aug_hmaps).mean(0).cpu().float().numpy(),
            torch.stack(aug_masks).mean(0).cpu().float().numpy(),
            torch.stack(aug_offsets).mean(0).cpu().float().numpy())


# ══════════════════════════════════════════════════════════════════
# 13.  GENERATE TEST VIDEO & RUN INFERENCE
# ══════════════════════════════════════════════════════════════════
print("\nGenerating test video ...")
FRAMES, DIFFUSION_D = 25, 2.5
positions = np.random.uniform(MAX_RADIUS + 2, IMAGE_SIZE - MAX_RADIUS - 2, (NUM_PARTICLES, 2)).astype(np.float32)
radii_vid = np.random.randint(MIN_RADIUS, MAX_RADIUS + 1, NUM_PARTICLES)
video_frames = []

for f in range(FRAMES):
    positions += np.random.normal(0, math.sqrt(2 * DIFFUSION_D), (NUM_PARTICLES, 2)).astype(np.float32)
    positions  = np.clip(positions, MAX_RADIUS + 2, IMAGE_SIZE - MAX_RADIUS - 2)
    render_type = _sample_render_type()
    bg          = make_background(IMAGE_SIZE, render_type)
    layers_vid  = []

    for i in range(NUM_PARTICLES):
        cx, cy   = int(positions[i, 0]), int(positions[i, 1])
        defocus, intens = random.uniform(0.15, 0.95), random.uniform(0.60, 1.0)
        asym, asym_ang  = random.uniform(0.0, 0.15), random.uniform(0.0, math.pi)
        sec_amp  = random.uniform(0.10, 0.40) if random.random() < 0.50 else 0.0
        result   = render_particle_layer((IMAGE_SIZE, IMAGE_SIZE), cx, cy, radii_vid[i], intens, defocus,
            asymmetry=asym, asymmetry_angle=asym_ang, secondary_fringe_amp=sec_amp, render_type=render_type)
        if result is not None: layers_vid.append(result)

    img = composite_layers(layers_vid, bg, render_type)
    img = cv2.GaussianBlur(img, (3, 3), 0.8)
    photon_scale = random.uniform(200.0, 600.0)
    photons      = np.clip(img * photon_scale, 0, 1e6).astype(np.float64)
    img          = np.random.poisson(photons).astype(np.float32) / photon_scale
    img         += np.random.normal(0, 0.015, img.shape).astype(np.float32)
    img          = np.clip(img, 0, None)
    if img.max() > img.min(): img = (img - img.min()) / (img.max() - img.min())
    video_frames.append(img)

print("Running inference (8-fold TTA) ...")
result_frames = []
for img in video_frames:
    tensor = torch.from_numpy(img[None, None]).to(device)
    hmap, mask, offset = infer_tta(tensor)
    centers = detect_centers(hmap, offset)
    result_frames.append({'hmap': hmap, 'mask': (mask > 0.5).astype(np.float32), 'centers': centers, 'offset': offset})


# ══════════════════════════════════════════════════════════════════
# 14.  TRAINING CURVE
# ══════════════════════════════════════════════════════════════════
fig_tc, (ax_l, ax_f) = plt.subplots(1, 2, figsize=(14, 4))
fig_tc.patch.set_facecolor('#0a0a0a')
for ax in (ax_l, ax_f):
    ax.set_facecolor('#111111')
    ax.tick_params(colors='#888888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333333')

ax_l.plot(train_losses, color='#00C8FF', lw=2, label='Train loss')
ax_l.plot(val_losses,   color='#FF6B6B', lw=2, label='Val loss')
ax_l.legend(facecolor='#1a1a1a', labelcolor='white')

ax_f.plot(val_f1s, color='#7CFF7C', lw=2, label='Val F1')
ax_f.set_ylim(0, 1)
ax_f.legend(facecolor='#1a1a1a', labelcolor='white')

fig_tc.suptitle('OptimalColloidNet v2 — Training Curves', color='white', fontsize=13)
plt.tight_layout()
curve_path = os.path.join(OUT_DIR, 'training_curve.png')
plt.savefig(curve_path, dpi=120, bbox_inches='tight', facecolor=fig_tc.get_facecolor())
plt.close(fig_tc)


# ══════════════════════════════════════════════════════════════════
# 15.  ANIMATION
# ══════════════════════════════════════════════════════════════════
print("\nRendering animation ...")
fig, (ax_in, ax_out) = plt.subplots(1, 2, figsize=(14, 7))
fig.patch.set_facecolor('#080808')
for ax in (ax_in, ax_out):
    ax.axis('off')
    ax.set_facecolor('#080808')

im_in     = ax_in.imshow(video_frames[0], cmap='gray', vmin=0, vmax=1)
im_out    = ax_out.imshow(result_frames[0]['mask'], cmap='gray', vmin=0, vmax=1)
scatter   = ax_out.scatter([], [], s=20, c='#00FFD0', edgecolors='white', linewidths=0.4, zorder=6)
frame_txt = ax_in.text(4,  12, '', color='#00FFD0', fontsize=9, fontfamily='monospace')
count_txt = ax_out.text(4, 12, '', color='#FFD700', fontsize=9, fontfamily='monospace')

def update(fi):
    rf = result_frames[fi]
    im_in.set_array(video_frames[fi])
    im_out.set_array(rf['mask'])
    pts = rf['centers']
    if len(pts): scatter.set_offsets(pts[:, [1, 0]])
    else:        scatter.set_offsets(np.empty((0, 2)))
    frame_txt.set_text(f'frame {fi:02d}/{FRAMES - 1}')
    count_txt.set_text(f'{len(pts)} particles')
    return [im_in, im_out, scatter, frame_txt, count_txt]

ani = animation.FuncAnimation(fig, update, frames=FRAMES, interval=220, blit=True)
plt.tight_layout(pad=1.5)
anim_path = os.path.join(OUT_DIR, 'result_animation.gif')
ani.save(anim_path, writer='pillow', fps=5)
plt.close(fig)
print(f"Animation saved -> {anim_path}")